# Chapter 5 — Methods and Methodologies
### Notebook 1 · Methodologies, and competency questions that actually run

*Book reference: Section 5.1*

Two halves. First: choosing a methodology is a decision with inputs, not a matter of taste. Second: the competency question — ontology engineering's vaguest artefact — becomes a pass/fail test.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch05_toolkit as ch5
from oe_course.sparql import SparqlStore
from oe_course.data import corpus
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

## 1. The catalogue

Each methodology was designed for a *situation*. The `fits when` column is the part that matters: it lists signals to look for in a project brief.

In [ ]:
print(pd.DataFrame(ch5.methodology_table()).to_string(index=False))

In [ ]:
for m in ch5.METHODOLOGIES:
    print(f'{m.name}')
    print(f'   phases: {" -> ".join(m.phases)}')
    print(f'   {m.note}\n')

## 2. Selecting from a brief

`recommend_methodology` scores each candidate against the brief and **returns the signals that matched**, so you can argue with it. A recommender that will not show its evidence is just an opinion with a UI.

In [ ]:
briefs = [
    'We must reuse existing ontologies and a legacy thesaurus for a networked project.',
    'Greenfield build from scratch by a single team, full lifecycle.',
    'A distributed consortium with many contributors, evolving over years.',
    'We want an agile, test driven, iterative approach in small increments.',
    'Knowledge management pilot; the business case first, application driven.',
]
for brief in briefs:
    r = ch5.recommend_methodology(brief)
    print(f"{r['recommended']:16s} <- matched {r['reason']}")
    print(f"{'':16s}    {brief}\n")

In [ ]:
r = ch5.recommend_methodology('We need an ontology.')
print('vague brief ->', r['recommended'])
print('reason      :', r['reason'])
print('\nA brief with no distinguishing signal gets the default. That is the\n'
      'honest answer -- and a hint that the brief needs more work before the\n'
      'methodology question can be answered at all.')

## 3. Competency questions as executable tests

A competency question states what the ontology must be able to answer. Written as prose it is unfalsifiable. Written as **SPARQL** it is a test: it either returns rows or it does not.

In [ ]:
for cq in ch5.AWO_CQS:
    print(f'[{cq.id}] {cq.question}')
    print(f'      needs: {cq.requires}')
    print('      ' + ' '.join(cq.sparql.split())[:100] + '\n')

In [ ]:
store = SparqlStore.in_memory(corpus.get('awo').turtle)
coverage = ch5.cq_coverage(ch5.AWO_CQS, store)
print(pd.DataFrame(coverage['results']).to_string(index=False))
print(f"\ncoverage = {coverage['coverage']}")

## 4. Coverage rises as the ontology is built

Here is the number that makes methodologies comparable. `ontology_at_stage` rebuilds the AWO from only the triples a given development step would have produced, and coverage is then **measured**, not asserted.

In [ ]:
stages = [set(), {'taxonomy'}, {'taxonomy', 'axioms'},
          {'taxonomy', 'axioms', 'instances'}]
rows = []
for done in stages:
    graph = ch5.ontology_at_stage(done)
    rows.append({'steps completed': ', '.join(sorted(done)) or '(nothing)',
                 'triples': len(graph),
                 'CQ coverage': ch5.coverage_for_steps(done)})
print(pd.DataFrame(rows).to_string(index=False))

> **Read the middle rows.** A taxonomy alone answers a third of the questions. Adding axioms takes it to five sixths — the axioms earn more coverage than the taxonomy did, which is the quantitative version of Chapter 1's argument that vocabulary without axioms is a word list. Instances add the last question and nothing else.

This table is what the Chapter 5 MDP is rewarded by.

### Exercise 1.1 — Write a competency question the AWO fails

Add a CQ that the finished AWO **cannot** answer, and show coverage dropping below 1.0. Then say what would have to be added to the ontology to satisfy it.

> **Hint.** Ask about something the ontology has no vocabulary for.

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution 1.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
extra = ch5.CompetencyQuestion(
    'cq7', 'Which animals are endangered?',
    'SELECT ?a WHERE { ?a rdfs:subClassOf awo:EndangeredSpecies }', 'axioms')
store = SparqlStore.in_memory(corpus.get('awo').turtle)
result = ch5.cq_coverage(list(ch5.AWO_CQS) + [extra], store)
print(f"coverage now {result['coverage']} ({result['answered']}/{result['total']})")
assert result['coverage'] < 1.0
print('\nThe AWO has no conservation-status vocabulary at all, so no query can\n'
      'answer this. Satisfying it means a new class and new axioms -- i.e. the\n'
      'CQ has just generated a requirement. That is what CQs are FOR: they are\n'
      'requirements written in a form that can fail.')

### Exercise 1.2 — Which step buys the most coverage per unit of effort?

Using `DEVELOPMENT_STEPS` costs and measured coverage, compute the coverage gained per unit cost for `taxonomy`, `axioms` and `instances`. Which is the best buy?

In [ ]:
# YOUR CODE HERE


<details>
<summary>Solution 1.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [ ]:
base = {'taxonomy'}
rows = []
for step, prior in [('taxonomy', set()),
                    ('axioms', {'taxonomy'}),
                    ('instances', {'taxonomy'})]:
    before = ch5.coverage_for_steps(prior)
    after = ch5.coverage_for_steps(prior | {step})
    cost = ch5.DEVELOPMENT_STEPS[step]['cost']
    rows.append({'step': step, 'coverage before': before, 'coverage after': after,
                 'gain': round(after - before, 3), 'cost': cost,
                 'gain per cost': round((after - before) / cost, 2)})
df = pd.DataFrame(rows)
print(df.to_string(index=False))
best = df.loc[df['gain per cost'].idxmax(), 'step']
print(f'\nbest buy: {best}')
assert best == 'axioms'
print('Axioms cost the most and still win on value per unit effort. The cheap\n'
      'step (instances) is the worst buy -- which is the opposite of what a team\n'
      'under deadline pressure usually does.')